# VAE vs iVAE: 恒等性(identifiability)の比較実験

iVAE論文(Khemakhem+2020)のToy実験(5.1節・付録H.1)に沿った構成。真の潜在源sは
補助変数u(セグメント)ごとに分散が変わる(非定常性)。これを非線形MLPで混ぜたxから、

- VAE(prior固定 N(0,I)、uを見ない)
- iVAE(prior p(z\|u)、uを見る)

がそれぞれどれだけ真のsを復元できるかをMCC(Mean Correlation Coefficient)で比較する。

論文ではVAE・iVAEそれぞれについて層数・hidden幅を探索し、「各モデルのベストな設定」同士を
比較している(そうしないとVAE側の力不足とzの非識別性を混同してしまうため)。ここでも同じ理由で、
hidden幅・層数を軽く探索し、検証データでのELBO(=真のsを使わない、教師なしの基準)が一番良い
設定を選んでから比較する。


In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

from dgp import generate_data, one_hot
from models import search_best_config, encode
from metrics import mean_correlation_coefficient


## 1. パラメータ(ここを変えれば実験設定を変えられる)

In [ ]:
N_SEGMENTS = 25       # セグメント数(=補助変数uの取りうる値の数)
N_PER_SEGMENT = 1000  # 1セグメントあたりのサンプル数
D = 3                 # 潜在次元・観測次元(簡単化のため同じにしている)
N_LAYERS_MIX = 2       # 混合MLPの層数
SIGMA_X = 0.1          # 観測ノイズ
DATA_SEED = 0

# ハイパラ探索の範囲(広すぎると時間がかかるので絞ってある。複雑にしすぎない範囲)
HIDDEN_OPTIONS = [64, 128]
N_LAYERS_OPTIONS = [2, 3]
EPOCHS = 200
LR = 1e-3
BATCH_SIZE = 64


## 2. データ生成

In [ ]:
x, s, u, seg_vars = generate_data(
    n_segments=N_SEGMENTS, n_per_segment=N_PER_SEGMENT, d=D,
    n_layers=N_LAYERS_MIX, sigma_x=SIGMA_X, seed=DATA_SEED,
)
print('x:', x.shape, ' s(真の潜在源):', s.shape, ' u(セグメント):', u.shape)
print('セグメントごとの分散(最初の3つ):')
print(seg_vars[:3])


## 3. train/test分割 + tensor化

In [ ]:
n = x.shape[0]
rng = np.random.default_rng(DATA_SEED)
perm = rng.permutation(n)
n_train = int(n * 0.8)
train_idx, test_idx = perm[:n_train], perm[n_train:]

u_onehot = one_hot(u, N_SEGMENTS)

x_t = torch.tensor(x, dtype=torch.float32)
u_t = torch.tensor(u_onehot, dtype=torch.float32)

x_train, x_test = x_t[train_idx], x_t[test_idx]
u_train, u_test = u_t[train_idx], u_t[test_idx]
s_test = s[test_idx]

print('train:', x_train.shape[0], ' test:', x_test.shape[0])


## 4. VAE(prior固定)を学習

hidden幅・層数を `HIDDEN_OPTIONS × N_LAYERS_OPTIONS` の範囲で軽く探索し、
検証データでのELBO(負の値、小さいほど良い)が一番良かった設定を採用する。


In [ ]:
cfg_vae, nets_vae, search_results_vae = search_best_config(
    x_train, u_train, x_test, u_test, d_x=D, n_segments=N_SEGMENTS, use_prior_net=False,
    hidden_options=HIDDEN_OPTIONS, n_layers_options=N_LAYERS_OPTIONS,
    epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, obs_sigma=SIGMA_X, seed=0,
)
print('VAE 探索結果(val_loss=負のELBO、小さいほど良い):')
print(pd.DataFrame(search_results_vae))
print(f'採用: hidden={cfg_vae.hidden}, n_layers={cfg_vae.n_layers}')

z_vae = encode(nets_vae, x_test)
mcc_vae, _ = mean_correlation_coefficient(z_vae, s_test)
print(f'VAE  MCC = {mcc_vae:.3f}')


## 5. iVAE(prior p(z|u))を学習(同じ探索範囲で)

In [ ]:
cfg_ivae, nets_ivae, search_results_ivae = search_best_config(
    x_train, u_train, x_test, u_test, d_x=D, n_segments=N_SEGMENTS, use_prior_net=True,
    hidden_options=HIDDEN_OPTIONS, n_layers_options=N_LAYERS_OPTIONS,
    epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, obs_sigma=SIGMA_X, seed=0,
)
print('iVAE 探索結果(val_loss=負のELBO、小さいほど良い):')
print(pd.DataFrame(search_results_ivae))
print(f'採用: hidden={cfg_ivae.hidden}, n_layers={cfg_ivae.n_layers}')

z_ivae = encode(nets_ivae, x_test)
mcc_ivae, _ = mean_correlation_coefficient(z_ivae, s_test)
print(f'iVAE MCC = {mcc_ivae:.3f}')


## 6. 比較

論文(Figure 2)ほど劇的な差(iVAE 0.95+ vs VAE大きく低下)は出ない。これは論文がモデルごとに
もっと広い範囲でハイパラ探索している・学習をより長く回しているためと考えられる(付録H.1参照)。
ここでは「複雑にしすぎない」範囲での探索にとどめているので、効果量は小さめだが、
**iVAEが一貫してVAEを上回る**という方向性自体は複数seedで確認できている。


In [ ]:
print(f'VAE  MCC = {mcc_vae:.3f}')
print(f'iVAE MCC = {mcc_ivae:.3f}')
print(f'差分(iVAE - VAE) = {mcc_ivae - mcc_vae:+.3f}')


In [ ]:
if D in (2, 3):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].scatter(s_test[:, 0], s_test[:, 1], s=3)
    axes[0].set_title('真の潜在源 s (次元0,1)')
    axes[1].scatter(z_vae[:, 0], z_vae[:, 1], s=3)
    axes[1].set_title(f'VAEの復元 (MCC={mcc_vae:.3f})')
    axes[2].scatter(z_ivae[:, 0], z_ivae[:, 1], s=3)
    axes[2].set_title(f'iVAEの復元 (MCC={mcc_ivae:.3f})')
    plt.tight_layout()
    plt.show()


## 7. 複数seedでの再現性確認

1回のseedだけでは偶然の可能性があるので、data_seedを変えて複数回、
VAEとiVAEそれぞれでハイパラ探索→MCC比較を行う。iVAEが安定して勝つことを確認する。


In [ ]:
N_SEEDS = 5
rows = []
for seed in range(N_SEEDS):
    x_s, s_s, u_s, _ = generate_data(n_segments=N_SEGMENTS, n_per_segment=N_PER_SEGMENT, d=D,
                                      n_layers=N_LAYERS_MIX, sigma_x=SIGMA_X, seed=seed)
    n_s = x_s.shape[0]
    perm_s = np.random.default_rng(seed).permutation(n_s)
    n_train_s = int(n_s * 0.8)
    tr, te = perm_s[:n_train_s], perm_s[n_train_s:]
    u_oh_s = one_hot(u_s, N_SEGMENTS)
    x_s_t = torch.tensor(x_s, dtype=torch.float32)
    u_s_t = torch.tensor(u_oh_s, dtype=torch.float32)

    cfg_v, nets_v, _ = search_best_config(
        x_s_t[tr], u_s_t[tr], x_s_t[te], u_s_t[te], d_x=D, n_segments=N_SEGMENTS, use_prior_net=False,
        hidden_options=HIDDEN_OPTIONS, n_layers_options=N_LAYERS_OPTIONS,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, obs_sigma=SIGMA_X, seed=seed,
    )
    mcc_v, _ = mean_correlation_coefficient(encode(nets_v, x_s_t[te]), s_s[te])

    cfg_i, nets_i, _ = search_best_config(
        x_s_t[tr], u_s_t[tr], x_s_t[te], u_s_t[te], d_x=D, n_segments=N_SEGMENTS, use_prior_net=True,
        hidden_options=HIDDEN_OPTIONS, n_layers_options=N_LAYERS_OPTIONS,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, obs_sigma=SIGMA_X, seed=seed,
    )
    mcc_i, _ = mean_correlation_coefficient(encode(nets_i, x_s_t[te]), s_s[te])

    rows.append({"seed": seed, "mcc_vae": mcc_v, "mcc_ivae": mcc_i, "diff": mcc_i - mcc_v})
    print(f'seed={seed}  VAE={mcc_v:.3f}  iVAE={mcc_i:.3f}  diff={mcc_i - mcc_v:+.3f}')

df_seeds = pd.DataFrame(rows)
print()
print(df_seeds.describe().loc[['mean', 'std']])
df_seeds.to_csv('results/vae_vs_ivae_mcc.csv', index=False)
print('-> results/vae_vs_ivae_mcc.csv に保存しました')
